# WRDS Extract 03: Fama-French Monthly Factors

This notebook extracts the monthly FF3 factors and saves them to parquet.

In [1]:
from pathlib import Path

import pandas as pd
import wrds
from pandas.tseries.offsets import MonthEnd

In [2]:
start_date = pd.Timestamp("2000-01-01")
end_date = pd.Timestamp("2025-12-31")
outdir = Path("data/raw")
ff_table = "ff.factors_monthly"

# Leave these as None if you want WRDS to use its default login flow.
wrds_username = None
wrds_password = None

outdir.mkdir(parents=True, exist_ok=True)
start_date, end_date, outdir

(Timestamp('2000-01-01 00:00:00'),
 Timestamp('2025-12-31 00:00:00'),
 PosixPath('data/raw'))

In [3]:
if wrds_username is not None and wrds_password is not None:
    db = wrds.Connection(wrds_username=wrds_username, wrds_password=wrds_password)
elif wrds_username is not None:
    db = wrds.Connection(wrds_username=wrds_username)
else:
    db = wrds.Connection()

db

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [4]:
ff_sql = f"""
    SELECT
        dateff::date AS dateff,
        mktrf,
        smb,
        hml,
        rf
    FROM {ff_table}
    WHERE dateff::date BETWEEN CAST(%(start_date)s AS date) AND CAST(%(end_date)s AS date)
    ORDER BY dateff
"""

ff = db.raw_sql(
    ff_sql,
    params={
        "start_date": start_date.date(),
        "end_date": end_date.date(),
    },
)

if ff.empty:
    raise RuntimeError(
        f"FF factors query returned zero rows from {ff_table!r}. "
        "The table may be unavailable for this WRDS account."
    )

ff.head()

,dateff,mktrf,smb,hml,rf
0,2000-01-31,-0.0474,0.0516,-0.0112,0.0041
1,2000-02-29,0.0246,0.2125,-0.0977,0.0043
2,2000-03-31,0.0521,-0.1741,0.085,0.0047
3,2000-04-28,-0.0639,-0.06,0.0645,0.0046
4,2000-05-31,-0.0439,-0.0608,0.0459,0.005


In [5]:
ff["dateff"] = pd.to_datetime(ff["dateff"], errors="coerce")
ff = ff.dropna(subset=["dateff"]).copy()
ff["month_end"] = ff["dateff"] + MonthEnd(0)

for col in ["mktrf", "smb", "hml", "rf"]:
    ff[col] = pd.to_numeric(ff[col], errors="coerce")

if ff.empty:
    raise RuntimeError("FF factor rows were returned, but none had valid dates.")

ff.head()

,dateff,mktrf,smb,hml,rf,month_end
0,2000-01-31,-0.0474,0.0516,-0.0112,0.0041,2000-01-31
1,2000-02-29,0.0246,0.2125,-0.0977,0.0043,2000-02-29
2,2000-03-31,0.0521,-0.1741,0.085,0.0047,2000-03-31
3,2000-04-28,-0.0639,-0.06,0.0645,0.0046,2000-04-30
4,2000-05-31,-0.0439,-0.0608,0.0459,0.005,2000-05-31


In [6]:
ff_outfile = outdir / "ff3_monthly.parquet"
ff.to_parquet(ff_outfile, engine="pyarrow", index=False)

print(f"saved: {ff_outfile}")
print(f"rows: {len(ff):,}")

saved: data/raw/ff3_monthly.parquet
rows: 312


In [7]:
db.close()